# 02 · Data preparation

This notebook runs the reusable pipeline in `src/data_pipeline.py`. Transformations preserve source-table grain, normalize malformed age groups, parse occurrence timestamps, and create a victim-level analysis table with a validated many-to-one join.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_pipeline import build_processed_data

summary = build_processed_data()
pd.Series(summary, name="rows")

incidents                23988
victims                  28748
offenders                18750
victim_event_rows        28748
unmatched_victim_rows        2
Name: rows, dtype: int64

## Validation checks

In [2]:
PROCESSED_DIR = ROOT / "data" / "processed"
shootings = pd.read_csv(PROCESSED_DIR / "shootings_clean.csv")
victims = pd.read_csv(PROCESSED_DIR / "shooting_victims_clean.csv")
offenders = pd.read_csv(PROCESSED_DIR / "shooting_offenders_clean.csv")
victim_events = pd.read_csv(PROCESSED_DIR / "shooting_victims_enriched.csv")

checks = {
    "incident IDs are unique": shootings["INCIDENT_KEY"].is_unique,
    "victim-level join preserves row count": len(victim_events) == len(victims),
    "fatal target contains only 0/1": set(victims["IS_FATAL"].dropna().unique()) <= {0, 1},
    "age groups are normalized": set(victims["VICTIM_AGE_GROUP"].unique()) <= {
        "<18", "18-24", "25-44", "45-64", "65+", "Unknown"
    },
}
assert all(checks.values())
pd.Series(checks, name="passed")

incident IDs are unique                  True
victim-level join preserves row count    True
fatal target contains only 0/1           True
age groups are normalized                True
Name: passed, dtype: bool

## Output contract

| File | Grain | Primary use |
|---|---|---|
| `shootings_clean.csv` | One row per incident | trends and seasonality |
| `shooting_victims_clean.csv` | One row per victim | victim outcomes |
| `shooting_offenders_clean.csv` | One row per recorded offender | separate offender analysis |
| `shooting_victims_enriched.csv` | One row per victim plus incident context | location analysis and modelling |

Two victim rows lack a matching occurrence timestamp in the incident extract. They remain in the processed table for traceability and are excluded only from analyses that require time.